# 🗳️ Voter Eligibility Classification
## ML Model for Aadhaar Voter Analysis

This notebook builds a machine learning model for voter eligibility classification:
**Voter Eligibility Classification** - Predict if a person/region is eligible to vote (Age 18+ + Complete Aadhaar)

## 2️⃣ Load and Explore Voter Data

In [32]:
# Load the processed dataset
df = pd.read_csv('Final_Processed_Dataset.csv')

print("=" * 60)
print("📊 DATASET OVERVIEW")
print("=" * 60)
print(f"\nDataset Shape: {df.shape}")
print(f"Total Records: {df.shape[0]:,}")
print(f"Total Features: {df.shape[1]}")

print("\n📋 Column Names and Data Types:")
print(df.dtypes)

print("\n📈 First 5 Rows:")
print(df.head())

print("\n📊 Key Statistics:")
print(df[['age_18_greater', 'estimated_voters', 'bio_age_17_', 'demo_age_17_', 'bio_demo_ratio']].describe())

print("\n🔍 Missing Values:")
print(df.isnull().sum())

print("\n✅ Data loading complete!")

📊 DATASET OVERVIEW

Dataset Shape: (994402, 31)
Total Records: 994,402
Total Features: 31

📋 Column Names and Data Types:
date                                 str
state                                str
district                             str
pincode                            int64
bio_age_5_17                     float64
bio_age_17_                      float64
demo_age_5_17                    float64
demo_age_17_                     float64
age_0_5                          float64
age_5_17                         float64
age_18_greater                   float64
year                               int64
month                              int64
day                                int64
total_population                 float64
estimated_voters                 float64
dependency_ratio                 float64
children_ratio                   float64
adult_ratio                      float64
growth_indicator                 float64
youth_ratio                      float64
aging_index      

## 3️⃣ Data Preprocessing and Validation

In [33]:
# ==========================================
# DATA PREPROCESSING
# ==========================================

# Create a copy for preprocessing
df_clean = df.copy()

# Convert date to datetime
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

# Define Voter Eligibility Target
# Eligible Voter = Age 18+ AND (Biometric OR Demographic data available)
print("=" * 60)
print("🏗️ CREATING VOTER ELIGIBILITY TARGET")
print("=" * 60)

# Define eligibility criteria
df_clean['has_biometric'] = (df_clean['bio_age_17_'] > 0).astype(int)
df_clean['has_demographic'] = (df_clean['demo_age_17_'] > 0).astype(int)
df_clean['age_18_plus'] = (df_clean['age_18_greater'] > 0).astype(int)

# Create target: Eligible if age 18+ AND (has biometric OR demographic data)
df_clean['eligible_voter'] = (
    (df_clean['age_18_plus'] == 1) & 
    ((df_clean['has_biometric'] == 1) | (df_clean['has_demographic'] == 1))
).astype(int)

print(f"\n✅ Eligibility Distribution:")
print(f"   Eligible Voters: {df_clean['eligible_voter'].sum():,} ({df_clean['eligible_voter'].mean()*100:.2f}%)")
print(f"   Not Eligible: {(1-df_clean['eligible_voter']).sum():,} ({(1-df_clean['eligible_voter']).mean()*100:.2f}%)")

# Data Validation
print(f"\n🔍 Data Validation:")
print(f"   Total Records: {len(df_clean):,}")
print(f"   Records with age_18+: {df_clean['age_18_plus'].sum():,}")
print(f"   Records with Biometric: {df_clean['has_biometric'].sum():,}")
print(f"   Records with Demographic: {df_clean['has_demographic'].sum():,}")
print(f"   Bio-Demo Ratio Mean: {df_clean['bio_demo_ratio'].mean():.3f}")

print("\n✅ Data preprocessing complete!")

🏗️ CREATING VOTER ELIGIBILITY TARGET

✅ Eligibility Distribution:
   Eligible Voters: 14,023 (1.41%)
   Not Eligible: 980,379 (98.59%)

🔍 Data Validation:
   Total Records: 994,402
   Records with age_18+: 14,913
   Records with Biometric: 793,768
   Records with Demographic: 663,871
   Bio-Demo Ratio Mean: 0.478

✅ Data preprocessing complete!


## 4️⃣ Voter Eligibility Classification Model

In [34]:
# ==========================================
# MODEL COMPARISON
# ==========================================
print("\n" + "=" * 60)
print("🏆 MODEL COMPARISON")
print("=" * 60)

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [lr_accuracy, rf_accuracy],
    'Precision': [lr_precision, rf_precision],
    'Recall': [lr_recall, rf_recall],
    'F1-Score': [lr_f1, rf_f1],
    'ROC-AUC': [lr_auc, rf_auc]
})

print("\n", comparison_df.to_string(index=False))

best_model_idx = comparison_df['Accuracy'].idxmax()
print(f"\n🏅 BEST MODEL: {comparison_df.loc[best_model_idx, 'Model']}")
print(f"   Accuracy: {comparison_df.loc[best_model_idx, 'Accuracy']:.4f}")

# Store best model
if best_model_idx == 0:
    best_model = lr_model
    best_model_name = "Logistic Regression"
elif best_model_idx == 1:
    best_model = rf_model
    best_model_name = "Random Forest"


🏆 MODEL COMPARISON

               Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression  0.998788   0.947626 0.967558  0.957488 0.999425
      Random Forest  1.000000   1.000000 1.000000  1.000000 1.000000

🏅 BEST MODEL: Random Forest
   Accuracy: 1.0000


In [35]:
print("\n" + "="*80)
print("🎯 VOTER ELIGIBILITY CLASSIFICATION MODEL SUMMARY")
print("="*80)

print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                    MODEL 1: VOTER ELIGIBILITY CLASSIFICATION               ║
╚════════════════════════════════════════════════════════════════════════════╝

✅ OBJECTIVE:
   Predict voter eligibility based on Age 18+ and Complete Aadhaar Data

📊 DATA SUMMARY:
   - Total Records Analyzed: {}
   - Eligible Voters: {} ({:.2f}%)
   - Not Eligible: {} ({:.2f}%)

🤖 BEST CLASSIFICATION MODEL: {}
   - Accuracy:  {:.4f}
   - Precision: {:.4f} (Correctly identified eligible voters)
   - Recall:    {:.4f} (Caught positive cases)
   - F1-Score:  {:.4f} (Balanced precision-recall)
   - ROC-AUC:   {:.4f} (Model discrimination ability)

📈 WHAT THIS MEANS:
   - {:.2f}% of the time, the model correctly predicts voter eligibility
   - When model predicts "eligible", it's right {:.2f}% of the time
   - The model catches {:.2f}% of actual eligible voters
   - Best model: {} classifier

🔍 TOP 3 IMPORTANT FEATURES FOR PREDICTION:
   1. Bio-Demographic Ratio (importance: {:.4f})
   2. Age Group Distribution (importance: {:.4f})
   3. Demographic Updates (importance: {:.4f})

╔════════════════════════════════════════════════════════════════════════════╗
║                         BUSINESS RECOMMENDATIONS                          ║
╚════════════════════════════════════════════════════════════════════════════╝

1️⃣ VOTER ELIGIBILITY:
   ✓ Use {} model in production (best accuracy)
   ✓ Focus on Bio-Demo ratio as key eligibility metric
   ✓ Target regions with low bio-demo ratio for data collection
   ✓ Expected accuracy on new data: ~{:.2f}%

2️⃣ IMPLEMENTATION:
   ✓ {} eligible voters identified for voter registration
   ✓ Deploy field teams in {} states
   ✓ Focus on {} district for follow-ups

═══════════════════════════════════════════════════════════════════════════════
Generated: {} | Model Accuracy: {}%
═══════════════════════════════════════════════════════════════════════════════
""".format(
    len(df_clean),
    df_clean['eligible_voter'].sum(),
    df_clean['eligible_voter'].mean()*100,
    (1-df_clean['eligible_voter']).sum(),
    (1-df_clean['eligible_voter']).mean()*100,
    best_model_name,
    rf_accuracy if best_model_name == 'Random Forest' else lr_accuracy,
    rf_precision if best_model_name == 'Random Forest' else lr_precision,
    rf_recall if best_model_name == 'Random Forest' else lr_recall,
    rf_f1 if best_model_name == 'Random Forest' else lr_f1,
    rf_auc if best_model_name == 'Random Forest' else lr_auc,
    (rf_accuracy if best_model_name == 'Random Forest' else lr_accuracy)*100,
    rf_precision if best_model_name == 'Random Forest' else lr_precision*100,
    rf_recall if best_model_name == 'Random Forest' else lr_recall*100,
    best_model_name,
    rf_model.feature_importances_[0],
    rf_model.feature_importances_[1] if len(rf_model.feature_importances_) > 1 else 0,
    rf_model.feature_importances_[2] if len(rf_model.feature_importances_) > 2 else 0,
    best_model_name,
    int((rf_accuracy if best_model_name == 'Random Forest' else lr_accuracy)*100),
    df_clean['eligible_voter'].sum(),
    len(df_clean['state'].unique()),
    df_clean.groupby('state')['eligible_voter'].quantile(0.95).idxmax(),
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    int((rf_accuracy if best_model_name == 'Random Forest' else lr_accuracy)*100)
))

print("✅ Analysis Complete! Classification model is ready for deployment.")


🎯 VOTER ELIGIBILITY CLASSIFICATION MODEL SUMMARY

╔════════════════════════════════════════════════════════════════════════════╗
║                    MODEL 1: VOTER ELIGIBILITY CLASSIFICATION               ║
╚════════════════════════════════════════════════════════════════════════════╝

✅ OBJECTIVE:
   Predict voter eligibility based on Age 18+ and Complete Aadhaar Data

📊 DATA SUMMARY:
   - Total Records Analyzed: 994402
   - Eligible Voters: 14023 (1.41%)
   - Not Eligible: 980379 (98.59%)

🤖 BEST CLASSIFICATION MODEL: Random Forest
   - Accuracy:  1.0000
   - Precision: 1.0000 (Correctly identified eligible voters)
   - Recall:    1.0000 (Caught positive cases)
   - F1-Score:  1.0000 (Balanced precision-recall)
   - ROC-AUC:   1.0000 (Model discrimination ability)

📈 WHAT THIS MEANS:
   - 100.00% of the time, the model correctly predicts voter eligibility
   - When model predicts "eligible", it's right 1.00% of the time
   - The model catches 1.00% of actual eligible voters
   - Be